# 07 - Seleccion final

Este notebook fija la decision final y valida el CSV que se entrega.


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'sample_submission.csv').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('No pude encontrar la raiz del repo')
    ROOT = ROOT.parent

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

pd.set_option('display.max_colwidth', 120)

RESULTS = ROOT / '07_final' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

selected = {
    'name': '3-way e100 weighted',
    'private_lb': 0.67126,
    'formula': '0.25*separable_headsep_e100_seed42 + 0.375*globalmel_sep_temporal_e100_seed42 + 0.375*sep_temporal_f1024_e100_seed42',
    'source_csv': 'investigation/results/submissions/parallel100_20260702/e100_headsep25_globalmel375_f1024_375.csv',
    'expected_sha256': '4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab',
}
selected


{'name': '3-way e100 weighted',
 'private_lb': 0.67126,
 'formula': '0.25*separable_headsep_e100_seed42 + 0.375*globalmel_sep_temporal_e100_seed42 + 0.375*sep_temporal_f1024_e100_seed42',
 'source_csv': 'investigation/results/submissions/parallel100_20260702/e100_headsep25_globalmel375_f1024_375.csv',
 'expected_sha256': '4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab'}

## Artefacto final

El CSV final se guarda en `07_final/submission.csv` y tambien en `100. Entregable/submission.csv`. Ambos deben coincidir con el CSV scoreado en Kaggle.


In [2]:
paths = {
    'source': ROOT / selected['source_csv'],
    '07_final': ROOT / '07_final' / 'submission.csv',
    '100_entregable': ROOT / '100. Entregable' / 'submission.csv',
}
rows = []
for name, path in paths.items():
    rows.append({'name': name, 'path': str(path.relative_to(ROOT)), 'exists': path.exists(), 'sha256': sha256(path) if path.exists() else None})
artifacts = pd.DataFrame(rows)
artifacts.to_csv(RESULTS / 'final_artifact_hashes.csv', index=False)
display(artifacts)
assert artifacts['exists'].all()
assert set(artifacts['sha256']) == {selected['expected_sha256']}


,name,path,exists,sha256
0,source,investigation/results/submissions/parallel100_20260702/e100_headsep25_globalmel375_f1024_375.csv,True,4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab
1,07_final,07_final/submission.csv,True,4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab
2,100_entregable,100. Entregable/submission.csv,True,4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab


In [3]:
sample = pd.read_csv(ROOT / 'data' / 'sample_submission.csv')
final_df = pd.read_csv(paths['07_final'])
label_columns = list(sample.columns[1:])
validation = {
    'rows': len(final_df),
    'columns': len(final_df.columns),
    'columns_match_sample': list(final_df.columns) == list(sample.columns),
    'fname_order_matches_sample': final_df['fname'].equals(sample['fname']),
    'min_probability': float(final_df[label_columns].min().min()),
    'max_probability': float(final_df[label_columns].max().max()),
    'probabilities_in_range': bool(final_df[label_columns].ge(0).all().all() and final_df[label_columns].le(1).all().all()),
}
(RESULTS / 'final_format_validation.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')
validation


{'rows': 3361,
 'columns': 81,
 'columns_match_sample': True,
 'fname_order_matches_sample': True,
 'min_probability': 3.113837592425e-11,
 'max_probability': 0.999998935,
 'probabilities_in_range': True}

## Por que no se selecciona el mejor score experimental

El `0.67674` con bagging es el mejor experimento registrado, pero agrega dos modelos extra dentro de las ramas temporales. Para la defensa se prioriza el `0.67126`, que mantiene tres componentes fisicos y una formula directa.


In [4]:
comparison = pd.DataFrame([
    {'candidate': '3-way e100 weighted', 'private_lb': 0.67126, 'selected_as_final': True, 'reason': 'tres componentes fisicos, pesos simples'},
    {'candidate': 'globalmel + f1024 bagged', 'private_lb': 0.67674, 'selected_as_final': False, 'reason': 'mejor experimento, pero con bagging interno'},
])
comparison.to_csv(RESULTS / 'final_vs_best_experiment.csv', index=False)
display(comparison)


,candidate,private_lb,selected_as_final,reason
0,3-way e100 weighted,0.67126,True,"tres componentes fisicos, pesos simples"
1,globalmel + f1024 bagged,0.67674,False,"mejor experimento, pero con bagging interno"


## Conclusion

La entrega oficial queda fijada en el `3-way e100 reponderado` con private LB `0.67126`. El CSV esta validado en formato, hash y equivalencia con el artefacto de `parallel100_20260702`.
